# CYGNO-30 internal-radioactivity background analysis — transparent notebook

This notebook is a **standalone, readable reimplementation of the current `study/analyze.py` Stage-B analysis** in `Oppedix/CYGNO_30_Sim` (current refactored repository). It assumes that the simulation campaign tarball has already been decompressed and that you can point the notebook to the extracted `campaign/` directory.

The goal is not to hide the analysis behind helper modules. The important logic is written explicitly in notebook cells so that you can inspect, modify, and rerun it block by block.

The pipeline reproduced here is:

```text
campaign metadata + 26 raw ROOT files
        ↓
strict campaign / ROOT validation
        ↓
stream Hits with uproot
        ↓
historical step-grouping algorithm
        ↓
electron/positron deposited energy per reconstructed group
        ↓
20 mm first-position fiducial cut
        ↓
activity × material quantity × seconds/year / generated primaries
        ↓
scale each physical contribution independently
        ↓
sum contributions into detector categories
        ↓
background spectra before / after cut
        ↓
Table 7.2-style totals and Table 7.3-style detector contributions
```

This corresponds to the internal-background calculation used for **Samuele's thesis Fig. 7.5, Fig. 7.6, Table 7.2 and Table 7.3**. It does **not** implement the later solar-neutrino signal templates, likelihood, toy Monte Carlo or Bayes-factor sensitivity analysis.

### Important historical conventions retained exactly

The notebook deliberately keeps the conventions used by the current repository analysis:

- 900 bins from 0 to 2000 keV.
- Raw Geant4 `EnergyDeposit` is stored in MeV; each per-volume value is multiplied by 1000 before summation.
- Only reconstructed groups finally labelled `e-` or `e+` enter the ER spectrum.
- `Nucleus` is the **most recently tracked ion label**, not a guaranteed per-hit ancestry relation.
- The grouping logic is the historical Samuele algorithm, including the `ionIoni` / `eIoni` continuation rule.
- Fiducialization uses the **first stored position in the first gas volume of the group**.
- The fiducial margin is exactly **20 mm** on x, y and z.
- Each source contribution is normalized **before** category summation.
- A year is exactly **365 days = 31,536,000 s**.
- Published thesis table values are comparison references only; they never rescale the Monte Carlo.

## 0. Requirements and campaign path

The decompressed archive normally contains a top-level directory named `campaign/` with files such as:

```text
campaign/
├── campaign.json
├── config.json
├── matrix.json
├── geometry.json
├── quantities.tsv
├── source/
├── probes/
└── jobs/
    ├── GEMsCore_U238/
    │   ├── manifest.json
    │   └── attempt-0001/
    │       ├── run.mac
    │       ├── simulation.log
    │       └── outfiles_V2/raw.root
    └── ... 26 contributions total
```

Python packages required for the analysis are the same basic scientific stack as the repository analysis: `uproot`, `awkward`, `numpy`, and `matplotlib`. The notebook itself also uses IPython display helpers that are already present in Jupyter.

Edit `CAMPAIGN_DIR` below so that it points to the directory that directly contains `campaign.json`.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import math
import re
from dataclasses import dataclass, field
from pathlib import PurePosixPath

import numpy as np
import uproot
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# -----------------------------------------------------------------------------
# USER SETTING: point this to the extracted directory containing campaign.json
# -----------------------------------------------------------------------------
CAMPAIGN_DIR = Path("./campaign").resolve()

# uproot memory budget used by the official analysis
STEP_SIZE = "64 MB"

# Current campaign contract: a full result contains all 26 source contributions.
ALLOW_PARTIAL = False

print("Campaign directory:", CAMPAIGN_DIR)
print("campaign.json exists:", (CAMPAIGN_DIR / "campaign.json").exists())

## 1. Small utilities used throughout the notebook

The official analysis is deliberately strict about provenance. Before looking at physics distributions it checks that files, manifests and accounting records still match the simulation campaign.

The functions below are copied conceptually from `study/runtime.py` and `study/source_matrix.py`, but kept here so that the notebook can run without importing the repository as a Python package.

In [ ]:
def require(condition, message):
    """Raise a readable error as soon as an analysis invariant is violated."""
    if not condition:
        raise ValueError(message)


def read_json(path):
    return json.loads(Path(path).read_text())


def sha256_file(path):
    """SHA-256 of a file, streamed in 1 MiB blocks."""
    path = Path(path)
    checksum = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            checksum.update(block)
    return checksum.hexdigest()


def object_digest(value):
    """Same canonical JSON hash convention used by the campaign runner."""
    payload = json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode()
    return hashlib.sha256(payload).hexdigest()


def positive(value):
    return type(value) in (int, float) and math.isfinite(value) and value > 0


def inside(root, name):
    """Reproduce the campaign path-safety check: no traversal and no symlinks."""
    root = Path(root)
    path = PurePosixPath(name)
    require(
        not path.is_absolute()
        and bool(path.parts)
        and ".." not in path.parts
        and "\\" not in name,
        "Unsafe campaign path: " + name,
    )
    result = root.joinpath(*path.parts)
    require(
        result.resolve().is_relative_to(root.resolve()) and not result.is_symlink(),
        "Unsafe campaign link",
    )
    return result

## 2. Load the campaign-level files first

Before opening ROOT, inspect the study identity. These files answer different questions:

- **`campaign.json`** — what exact simulation campaign was run, which jobs completed, environment/build/source identity, and generated-primary accounting.
- **`config.json`** — run-level settings such as layout and primaries per contribution.
- **`matrix.json`** — the 26 radioactive source contributions and their assay activities.
- **`quantities.tsv`** — masses or piece counts derived from the constructed detector geometry.
- **`geometry.json`** — gas dimensions and center coordinates used for fiducialization.

The numerical analysis should be understood as combining `matrix.json` (radioactivity) with `quantities.tsv` (how much material exists) and each ROOT file (how often a simulated decay produces an accepted ER group).

In [ ]:
campaign = read_json(CAMPAIGN_DIR / "campaign.json")
config = read_json(CAMPAIGN_DIR / "config.json")
matrix = read_json(CAMPAIGN_DIR / "matrix.json")
geometry = read_json(CAMPAIGN_DIR / "geometry.json")

print("Campaign stage:", campaign.get("stage"))
print("Campaign status:", campaign.get("status"))
print("Coverage:", campaign.get("coverage"))
print("Mode:", config.get("mode"))
print("Layout:", config.get("layout"))
print("Model:", config.get("model"))
print("Source policy:", config.get("source_policy"))
print("Primaries requested per contribution:", config.get("primaries_per_job"))
print("Number of matrix rows:", len(matrix.get("contributions", [])))
print("Gas size [mm]:", geometry.get("gas_size_mm"))
print("Number of gas centers:", len(geometry.get("gas_centers_mm", {})))

## 3. Validate the radioactive-source matrix and constructed quantities

The matrix is the machine-readable version of the background-source definition. A contribution is not just an isotope name; it specifies:

- the physical component in which the parent nucleus is generated;
- the analysis category to which it will eventually contribute;
- assay activity and units;
- isotope `(Z, A)`;
- whether the chain is complete or split at a daughter boundary.

The analysis has **26 contributions** because materials with different assay values must be simulated and normalized separately before they can be added together.

In [ ]:
LAYOUT_MODULE_COUNTS = {
    "legacy-25x3": 75,
    "cygno-5x5x3-v1": 75,
}
LAYOUTS = set(LAYOUT_MODULE_COUNTS)

COMPONENTS = {
    "GEMsCore":     ("Acrylic", "GEMs"),
    "GEMsOuter":    ("EFCu", "GEMs"),
    "RingSupports": ("Acrylic", "Field Cage"),
    "RingStrips":   ("EFCu", "Field Cage"),
    "Cathodes":     ("EFCu", "Cathodes"),
    "Vessel":       ("EFCu", "Vessel"),
    "Lens":         ("Suprasil", "Camera Lenses"),
    "Sensors":      ("Silicon", "Camera Sensors"),
    "Resistors":    ("Al2O3", "Resistors"),
}

IONS = {
    "U238": (92, 238),
    "Th232": (90, 232),
    "K40": (19, 40),
    "U235": (92, 235),
    "Ra226": (88, 226),
    "Th228": (90, 228),
}


def ion_name(ion):
    require(isinstance(ion, dict), "Missing nucleus record")
    name = ion.get("name")
    require(
        name in IONS and (ion.get("Z"), ion.get("A")) == IONS[name],
        "Invalid nucleus Z/A",
    )
    return name


def validate_matrix(matrix):
    require(matrix.get("schema_version") == 1, "Unsupported matrix schema")
    require(matrix.get("matrix_id") == "thesis-table7.1-v1", "Unsupported source matrix")
    require(matrix.get("seconds_per_year") == 31536000, "Study year convention changed")
    require(
        matrix.get("quantity_policy") == "constructed-component-mass-kg-or-placement-count",
        "Invalid quantity policy",
    )

    provenance = matrix.get("provenance", {})
    require(
        provenance.get("table") == "7.1" and len(provenance.get("sha256", "")) == 64,
        "Missing Table 7.1 provenance",
    )

    rows = matrix.get("contributions", [])
    seen = set()

    for row in rows:
        component = row.get("component")
        require(component in COMPONENTS, "Unknown/ambiguous source component")

        material, category = COMPONENTS[component]
        require(
            (row.get("assay_material"), row.get("category")) == (material, category),
            "Wrong assay/category mapping",
        )

        isotope = ion_name(row.get("chain_start"))
        key = (component, isotope)
        require(
            key not in seen and row.get("id") == "_".join(key),
            "Duplicate or malformed contribution ID",
        )
        seen.add(key)

        require(positive(row.get("activity")), "Activity must be finite and positive")
        expected_unit = "Bq/piece" if component == "Resistors" else "Bq/kg"
        require(row.get("activity_unit") == expected_unit, "Wrong activity unit")

        expected_basis = "upper-limit-used-as-value" if material == "Acrylic" else "table-value"
        require(row.get("activity_basis") == expected_basis, "Wrong activity interpretation")
        require(row.get("full_chain") is True, "Chain transport must be enabled")

        stop = row.get("stop_before")
        expected_stop = (
            {"U238": "Ra226", "Th232": "Th228"}.get(isotope)
            if component == "Resistors"
            else None
        )
        actual_stop = ion_name(stop) if stop is not None else None
        require(actual_stop == expected_stop, "Wrong stop-before boundary")

        policy = (
            "upper-segment" if expected_stop else
            "lower-segment" if isotope in ("Ra226", "Th228") else
            "single-parent" if isotope == "K40" else
            "equilibrium"
        )
        require(row.get("chain_policy") == policy, "Wrong chain policy")

        daughters = (
            [{"U238": "Ra226", "Th232": "Th228"}[isotope]]
            if component != "Resistors" and isotope in ("U238", "Th232")
            else []
        )
        require(
            row.get("equilibrium_daughters") == daughters,
            "Equilibrium daughters must not become separate jobs",
        )

    expected = {
        (component, isotope)
        for component, (material, _) in COMPONENTS.items()
        for isotope in (
            IONS if component == "Resistors" else
            ("U238", "Th232") if material == "EFCu" else
            ("U238", "Th232", "K40")
        )
    }
    require(seen == expected and len(rows) == 26, "Expected exactly 26 contributions")
    return matrix


matrix = validate_matrix(matrix)
print("Validated matrix:", matrix["matrix_id"], "with", len(matrix["contributions"]), "contributions")

In [ ]:
def read_quantities(path, *, layout, model, geometry_hash):
    """Read the geometry-derived mass/piece table exactly as Stage B does."""
    require(layout in LAYOUTS and model == "code-compatible", "Unsupported constructed detector")

    lines = Path(path).read_text().splitlines()
    headers = {}
    for line in lines:
        if line.startswith("# "):
            key, value = line[2:].split(":", 1)
            require(key not in headers, "Duplicate quantities metadata")
            headers[key] = value.strip()

    expected_headers = {
        "layout": layout,
        "detector-model": model,
        "source-policy": "historical",
        "geometry-hash": geometry_hash,
    }
    for key, expected in expected_headers.items():
        require(headers.get(key) == expected, f"Quantities {key} mismatch")

    require(headers.get("provenance"), "Missing constructed quantities provenance")

    result = {}
    data_lines = (line for line in lines if not line.startswith("#"))
    for row in csv.DictReader(data_lines, delimiter="\t"):
        component = row["component"]
        require(component in COMPONENTS and component not in result, "Unknown/duplicate quantity component")
        mass = float(row["mass_kg"])
        pieces = int(row["pieces"])
        require(positive(mass) and pieces > 0 and row["geometry_material"], "Invalid constructed quantity")
        result[component] = {
            "mass_kg": mass,
            "pieces": pieces,
            "geometry_material": row["geometry_material"],
        }

    require(result.keys() == COMPONENTS.keys(), "Incomplete constructed quantities")
    return result


def quantity_for(row, quantities):
    """Use mass for Bq/kg contributions and placement count for Bq/piece contributions."""
    unit = row["activity_unit"]
    require(unit in ("Bq/kg", "Bq/piece"), "Unsupported activity unit")
    key = "mass_kg" if unit == "Bq/kg" else "pieces"
    return quantities[row["component"]][key]


env = campaign["identity"]["effective_environment"]
quantities = read_quantities(
    CAMPAIGN_DIR / "quantities.tsv",
    layout=config["layout"],
    model=config["model"],
    geometry_hash=env["geometry_hash"],
)

for component, q in quantities.items():
    print(f"{component:14s}  mass={q['mass_kg']:12.6g} kg   pieces={q['pieces']:5d}   material={q['geometry_material']}")

## 4. Inspect the 26 physical contributions before reading any ROOT data

The table below is useful because it separates three concepts that are easy to mix up:

1. **component** — the actual Geant4 source object (`GEMsCore`, `RingStrips`, ...);
2. **category** — the final thesis plotting category (`GEMs`, `Field Cage`, ...);
3. **activity** — the assay value used to normalize this contribution.

For `Bq/kg` sources the normalization quantity is the constructed component mass. For resistors, whose assay is `Bq/piece`, the quantity is the number of placed resistor pieces.

In [ ]:
def markdown_table(rows, columns, formatters=None, max_rows=None):
    """Small dependency-free table renderer for notebook display."""
    formatters = formatters or {}
    rows_to_show = rows if max_rows is None else rows[:max_rows]
    header = "| " + " | ".join(columns) + " |"
    sep = "|" + "|".join(["---"] * len(columns)) + "|"
    lines = [header, sep]
    for row in rows_to_show:
        vals = []
        for col in columns:
            value = row.get(col, "")
            if col in formatters:
                value = formatters[col](value)
            vals.append(str(value))
        lines.append("| " + " | ".join(vals) + " |")
    if max_rows is not None and len(rows) > max_rows:
        lines.append(f"\n_Showing {max_rows} of {len(rows)} rows._")
    return "\n".join(lines)


source_rows = []
for row in matrix["contributions"]:
    quantity = quantity_for(row, quantities)
    source_rows.append({
        "id": row["id"],
        "component": row["component"],
        "category": row["category"],
        "isotope": row["chain_start"]["name"],
        "activity": row["activity"],
        "activity_unit": row["activity_unit"],
        "quantity": quantity,
        "chain_policy": row["chain_policy"],
        "stop_before": row["stop_before"]["name"] if row["stop_before"] else "—",
    })


display(Markdown(markdown_table(
    source_rows,
    ["id", "category", "activity", "activity_unit", "quantity", "chain_policy", "stop_before"],
    formatters={"activity": lambda x: f"{x:.6g}", "quantity": lambda x: f"{x:.6g}"},
)))

## 5. Strict campaign validation

The original `study/analyze.py` calls `validate_campaign(...)` before performing any numerical analysis. The following cells reproduce that integrity layer explicitly.

This validation is intentionally heavy because it verifies checksums for the source snapshot and every attempt artifact. For a large campaign this means significant disk I/O, but it is part of the current analysis contract.

If you are developing interactively, run this validation once after extracting the archive; then you can modify later analysis cells without repeating it every time.

In [ ]:
RDM_COMMAND = "/process/had/rdm/thresholdForVeryLongDecayTime 1.0e+60 year"
RDM_SECONDS = 1.0e60 * 31536000

ENV_KEYS = (
    "G4NEUTRONHPDATA", "G4LEDATA", "G4LEVELGAMMADATA", "G4RADIOACTIVEDATA",
    "G4PARTICLEXSDATA", "G4PIIDATA", "G4REALSURFACEDATA", "G4SAIDXSDATA",
    "G4ABLADATA", "G4INCLDATA", "G4ENSDFSTATEDATA", "G4CHANNELINGDATA",
)

ACCOUNTING_FIELDS = (
    "RunID", "RequestedEvents", "GeneratedPrimaries", "ProcessedEvents", "AbortedEvents"
)


def decay_commands(row):
    start = row["chain_start"]
    stop = row["stop_before"] or {"Z": 0, "A": 0}
    return [
        f"/detector/RadElement {row['component']}",
        f"/isotope/AtomicNumber {start['Z']}",
        f"/isotope/MassNumber {start['A']}",
        "/rdecay01/fullChain true",
        f"/stopChain/ZStopDecay {stop['Z']}",
        f"/stopChain/AStopDecay {stop['A']}",
    ]


def seeds_for(seed, contribution):
    value = hashlib.sha256(f"{seed}:{contribution}".encode()).digest()
    return [
        int.from_bytes(value[i:i+8], "big") % 2_000_000_000 + 1
        for i in (0, 8)
    ]


def macro_for(row, config):
    seeds = seeds_for(config["seed"], row["id"])
    lines = [
        f"# {row['id']}: {config['mode']}; code-compatible / historical",
        RDM_COMMAND,
        "/run/printProgress 10000",
        "/run/verbose 0",
        "/event/verbose 0",
        "/tracking/verbose 0",
        "/random/setSeeds " + " ".join(map(str, seeds)),
        "/output/OutFile raw",
        *decay_commands(row),
        f"/run/beamOn {config['primaries_per_job']}",
        "",
    ]
    return "\n".join(lines)


def validate_accounting(values, requested):
    require(
        set(values) == set(ACCOUNTING_FIELDS)
        and all(type(x) is int for x in values.values()),
        "Invalid accounting schema",
    )
    expected = {
        "RunID": 0,
        "RequestedEvents": requested,
        "GeneratedPrimaries": requested,
        "ProcessedEvents": requested,
        "AbortedEvents": 0,
    }
    require(values == expected, f"Incomplete/aborted run: {values}; requested {requested}")
    return values


def parse_environment(log):
    values = {}
    for line in log.splitlines():
        if line.startswith("CYGNO_ENV "):
            _, key, value = line.split(" ", 2)
            require(key not in values, "Duplicate environment marker")
            values[key] = value

    required = {*ENV_KEYS, "source_hash", "geometry_hash", "geant4", "physics", "radioactive_decay_time_threshold_s"}
    require(
        set(values) == required and all(values[k] != "UNRESOLVED" for k in ENV_KEYS),
        "Missing effective physics/data environment",
    )
    require(values["physics"] == "QGSP_BIC_EMZ+G4RadioactiveDecayPhysics", "Unexpected physics")
    return values


def parse_accounting(log, requested):
    lines = re.findall(r"^(?:G4WT0 > )?CYGNO_ACCOUNTING (.+)$", log, re.M)
    require(len(lines) == 1, "Need exactly one CYGNO_ACCOUNTING record")

    def unique(pairs):
        result = {}
        for key, value in pairs:
            require(key not in result, "Duplicate accounting field")
            result[key] = value
        return result

    values = json.loads(lines[0], object_pairs_hook=unique)
    return validate_accounting(values, requested)


def validate_simulation_log(log, environment, requested):
    forbidden = (
        "FatalException", "PART122", "COMMAND NOT FOUND", "***** Illegal",
        "RunAborted", "EventAborted",
    )
    require(
        not any(token in log for token in forbidden)
        and not re.search(r"\b(?:run|event)\b[^\n]*\babort(?:ed|ing)\b", log, re.I),
        "Fatal/aborted simulation",
    )
    require(parse_environment(log) == environment, "Runtime environment changed")
    thresholds = re.findall(r"CYGNO_RUN radioactive_decay_time_threshold_s (\S+)", log)
    require(
        len(thresholds) == 1
        and math.isclose(float(thresholds[0]), RDM_SECONDS, rel_tol=1e-12),
        "Worker did not use required radioactive-decay threshold",
    )
    return parse_accounting(log, requested)


def artifacts(directory):
    directory = Path(directory)
    return {
        str(p.relative_to(directory)): sha256_file(p)
        for p in sorted(directory.rglob("*"))
        if p.is_file() and p.name != "manifest.json"
    }


def check_artifacts(directory, manifest):
    require(manifest["artifacts"] == artifacts(directory), f"Artifact mismatch: {directory}")

In [ ]:
def validate_snapshots(output, campaign):
    identity = campaign["identity"]
    require(campaign["fingerprint"] == object_digest(identity), "Campaign fingerprint mismatch")
    require(
        set(campaign["snapshots"]) == {"config.json", "matrix.json", "geometry.json", "quantities.tsv"},
        "Missing campaign snapshots",
    )

    for name, checksum in campaign["snapshots"].items():
        require(sha256_file(inside(output, name)) == checksum, "Snapshot checksum mismatch: " + name)

    require(
        read_json(output / "config.json") == identity["config"]
        and read_json(output / "matrix.json") == identity["matrix"],
        "Configuration/matrix identity mismatch",
    )

    # The archive contains the exact source snapshot that generated the campaign.
    for name, checksum in identity["source"]["files_sha256"].items():
        if checksum != "missing":
            require(
                sha256_file(inside(output / "source", name)) == checksum,
                "Source snapshot mismatch: " + name,
            )

    cfg = identity["config"]
    env = identity["effective_environment"]
    q = read_quantities(
        output / "quantities.tsv",
        layout=cfg["layout"],
        model=cfg["model"],
        geometry_hash=env["geometry_hash"],
    )
    require(q == campaign["quantities"], "Quantity mismatch")

    required_source_hash_line = f"# compiled-source-hash: {env['source_hash']}"
    require(
        required_source_hash_line in (output / "quantities.tsv").read_text().splitlines(),
        "Quantity source hash mismatch",
    )

    g = read_json(output / "geometry.json")
    require(
        g["layout"] == cfg["layout"]
        and g["geometry_hash"] == env["geometry_hash"]
        and g["source_hash"] == env["source_hash"],
        "Geometry identity mismatch",
    )

    expected_gas_ids = {
        str(i) for i in range(2 * LAYOUT_MODULE_COUNTS[cfg["layout"]])
    }
    require(set(g["gas_centers_mm"]) == expected_gas_ids, "Invalid gas copy map")

    for values in [g["gas_size_mm"], *g["gas_centers_mm"].values()]:
        require(len(values) == 3 and all(math.isfinite(v) for v in values), "Invalid geometry coordinates")
    require(all(v > 40 for v in g["gas_size_mm"]), "Invalid fiducial dimensions")


def validate_preflight(output, campaign):
    path = inside(output, campaign["preflight"])
    check_artifacts(path.parent, {"artifacts": campaign["preflight_artifacts"]})
    preflight = read_json(path)

    names = {"u238-default", "u238-enabled", "th232-full", "bi212-full"}
    require(
        set(preflight["checks"]) == names
        and preflight["complete"]
        and all(c["status"] == "passed" and not c["PART122"] for c in preflight["checks"].values()),
        "Four-case decay preflight failed",
    )

    for name, check in preflight["checks"].items():
        expected = campaign["identity"]["effective_environment"].copy()
        if name == "u238-default":
            expected["radioactive_decay_time_threshold_s"] = check["environment"]["radioactive_decay_time_threshold_s"]
        require(check["environment"] == expected, "Preflight environment mismatch")


def validate_job(output, row, campaign):
    config = campaign["identity"]["config"]
    directory = output / "jobs" / row["id"]
    job = read_json(directory / "manifest.json")

    require(
        job["contribution"] == row["id"]
        and job["campaign_fingerprint"] == campaign["fingerprint"],
        "Mixed contribution/campaign identity",
    )

    macro = macro_for(row, config)
    expected_fingerprint = object_digest({
        "campaign": campaign["fingerprint"],
        "contribution": row,
        "macro": macro,
    })
    require(job["fingerprint"] == expected_fingerprint, "Job fingerprint mismatch")
    require(job["status"] == "complete", "Incomplete job")

    attempt = inside(directory, job["attempt"])
    require(read_json(attempt / "manifest.json") == job, "Attempt manifest mismatch")
    check_artifacts(attempt, job)

    require(
        (attempt / "run.mac").read_text() == macro
        and job["macro_sha256"] == sha256_file(attempt / "run.mac"),
        "Macro mismatch",
    )

    require(
        job["seeds"] == seeds_for(config["seed"], row["id"])
        and job["workers"] == 1
        and job["requested_primaries"] == config["primaries_per_job"],
        "Job configuration mismatch",
    )

    receipt = read_json(attempt / "simulation.command.json")
    require(
        receipt.get("returncode") == 0
        and not receipt.get("timed_out")
        and not receipt.get("interrupted"),
        "Incomplete simulation receipt",
    )

    accounting = validate_simulation_log(
        (attempt / "simulation.log").read_text(errors="replace"),
        campaign["identity"]["effective_environment"],
        config["primaries_per_job"],
    )
    require(
        job["accounting"] == accounting
        and job["generated_primaries"] == accounting["GeneratedPrimaries"],
        "Manifest/log accounting mismatch",
    )

    raw = inside(attempt, job["raw_path"])
    require(
        raw.name in ("raw.root", "raw_t0.root")
        and raw.stat().st_size > 0
        and list((attempt / "outfiles_V2").glob("*.root")) == [raw],
        "Invalid raw output",
    )
    return job, raw


def validate_campaign(output, allow_partial=False):
    campaign = read_json(output / "campaign.json")
    require(
        campaign["schema_version"] == 2 and campaign["stage"] == "simulation",
        "Not a stage A campaign",
    )

    validate_snapshots(output, campaign)
    validate_preflight(output, campaign)
    matrix = validate_matrix(read_json(output / "matrix.json"))

    jobs = {}
    records = campaign["job_records"]
    require(
        set(records) == {r["id"] for r in matrix["contributions"]},
        "Invalid campaign job coverage",
    )

    for row in matrix["contributions"]:
        path = output / "jobs" / row["id"] / "manifest.json"
        job = read_json(path) if path.exists() else {"status": "not_started"}
        require(job == records[row["id"]], "Campaign/job manifest disagreement")
        if job["status"] == "complete":
            jobs[row["id"]] = validate_job(output, row, campaign)

    require(
        campaign["coverage"] == f"{len(jobs)}/26"
        and (campaign["status"] == "complete") == (len(jobs) == 26),
        "Campaign completeness mismatch",
    )
    require(
        allow_partial or len(jobs) == 26,
        "Incomplete campaign; partial diagnostics require ALLOW_PARTIAL=True",
    )
    return jobs

In [ ]:
# This is intentionally the slow, strict validation used before numerical analysis.
jobs = validate_campaign(CAMPAIGN_DIR, allow_partial=ALLOW_PARTIAL)

print(f"Validated complete jobs: {len(jobs)}/26")
for contribution_id in list(jobs)[:5]:
    job, raw_path = jobs[contribution_id]
    print(f"  {contribution_id:24s} -> {raw_path.relative_to(CAMPAIGN_DIR)}")
if len(jobs) > 5:
    print("  ...")

## 6. Understand one raw ROOT file before processing all 26

Each simulation output contains three TTrees:

### `Hits`
One row is one Geant4 step in a sensitive gas volume. The important branches are:

| Branch | Meaning |
|---|---|
| `EventNumber` | Geant4 event number |
| `ParticleName` | particle making the step |
| `ParticleID` | event-local Geant4 track ID, **not PDG code** |
| `ParentID` | Geant4 parent track ID |
| `x_hits`, `y_hits`, `z_hits` | pre-step global position in mm |
| `EnergyDeposit` | energy deposited in this step, MeV |
| `VolumeNumber` | sensitive-gas copy number |
| `Nucleus` | most recently tracked ion label |
| `ProcessType` | creator process of the track |

### `RunMetadata`
A one-row identity record: geometry hash, layout, detector model and source policy.

### `RunAccounting`
A one-row run-completeness record: requested/generated/processed events and aborted events.

The raw `Hits` tree is **not yet an electron-recoil spectrum**. One physical electron can leave many rows, possibly in more than one gas volume. The grouping stage below reconstructs the historical analysis-level object.

In [ ]:
HITS_SCHEMA = {
    "EventNumber": "int32",
    "ParticleName": "string",
    "ParticleID": "int32",
    "ParticleTag": "int32",
    "ParentID": "int32",
    "x_hits": "float64",
    "y_hits": "float64",
    "z_hits": "float64",
    "EnergyDeposit": "float64",
    "VolumeNumber": "int32",
    "Nucleus": "string",
    "ProcessType": "string",
}

METADATA = {
    "GeometryHash": "geometry_hash",
    "Layout": "layout",
    "DetectorModel": "model",
    "SourcePolicy": "source_policy",
}

ROOT_TYPES = {
    "int32": {"int32_t", "int"},
    "float64": {"double"},
    "string": {"char*"},
}


def validate_tree(file, name, schema, one=False):
    require(name in file and file[name].classname == "TTree", "Missing/non-TTree " + name)
    result = file[name]
    require(set(result.keys()) == set(schema), "Invalid " + name + " field names")
    for key, kind in schema.items():
        require(
            result[key].typename in ROOT_TYPES[kind],
            f"Invalid {name}.{key} type: {result[key].typename}",
        )
    if one:
        require(result.num_entries == 1, "Need exactly one " + name + " row")
    return result


def root_header(file, expected, requested):
    metadata_tree = validate_tree(
        file,
        "RunMetadata",
        dict.fromkeys(METADATA, "string"),
        one=True,
    )
    metadata = metadata_tree.arrays(library="np")
    for field, key in METADATA.items():
        require(metadata[field][0] == expected[key], "Raw identity mismatch: " + field)

    accounting_tree = validate_tree(
        file,
        "RunAccounting",
        dict.fromkeys(ACCOUNTING_FIELDS, "int32"),
        one=True,
    )
    accounting_arrays = accounting_tree.arrays(library="np")
    accounting = {name: int(values[0]) for name, values in accounting_arrays.items()}
    validate_accounting(accounting, requested)

    hits = validate_tree(file, "Hits", HITS_SCHEMA)
    return accounting, hits

In [ ]:
# Pick the first contribution only for didactic inspection.
example_row = matrix["contributions"][0]
example_job, example_raw = jobs[example_row["id"]]

expected_identity = {
    key: config[key] for key in ("layout", "model", "source_policy")
}
expected_identity["geometry_hash"] = env["geometry_hash"]

with uproot.open(example_raw, array_cache=None) as f:
    print("ROOT file:", example_raw)
    print("Objects:", list(f.keys()))
    accounting, hits = root_header(
        f,
        expected_identity,
        example_job["requested_primaries"],
    )
    print("\nRun accounting:")
    print(accounting)
    print("\nHits entries:", hits.num_entries)
    print("\nHits schema:")
    for branch in hits.keys():
        print(f"  {branch:16s} {hits[branch].typename}")

    # Small sample only: this does not load the full ROOT file.
    sample = hits.arrays(entry_start=0, entry_stop=min(8, hits.num_entries), library="np")

sample_rows = []
for i in range(len(sample["EventNumber"])):
    sample_rows.append({
        "EventNumber": int(sample["EventNumber"][i]),
        "ParticleName": sample["ParticleName"][i],
        "ParticleID": int(sample["ParticleID"][i]),
        "ParentID": int(sample["ParentID"][i]),
        "EnergyDeposit": float(sample["EnergyDeposit"][i]),
        "VolumeNumber": int(sample["VolumeNumber"][i]),
        "Nucleus": sample["Nucleus"][i],
        "ProcessType": sample["ProcessType"][i],
    })

display(Markdown("### First few raw `Hits` rows\n\n" + markdown_table(
    sample_rows,
    ["EventNumber", "ParticleName", "ParticleID", "ParentID", "EnergyDeposit", "VolumeNumber", "Nucleus", "ProcessType"],
    formatters={"EnergyDeposit": lambda x: f"{x:.6g}"},
)))

## 7. Stream and validate the `Hits` tree

The production files can be large, so the official analysis does not load all rows into memory. `uproot.iterate` reads chunks with a memory budget (`64 MB` by default).

For every chunk we check:

- event IDs are valid and non-decreasing;
- coordinates and energy deposits are finite;
- gas copy numbers are inside the valid layout range.

For the `legacy-25x3` profile there are 75 back-to-back modules and therefore **150 sensitive gas volumes**, numbered 0–149.

In [ ]:
def hit_chunks(hits, requested, layout, step_size=STEP_SIZE):
    last_event = -1

    for chunk in hits.iterate(step_size=step_size, library="np"):
        events = chunk["EventNumber"]
        if not len(events):
            continue

        require(np.all((events >= 0) & (events < requested)), "Invalid event ID")
        require(
            events[0] >= last_event and np.all(events[1:] >= events[:-1]),
            "Out-of-order single-worker events",
        )
        last_event = int(events[-1])

        for field_name in ("x_hits", "y_hits", "z_hits", "EnergyDeposit"):
            require(np.all(np.isfinite(chunk[field_name])), "Nonfinite Hits " + field_name)

        volumes = chunk["VolumeNumber"]
        require(
            np.all((volumes >= 0) & (volumes < 2 * LAYOUT_MODULE_COUNTS[layout])),
            "Invalid gas copy",
        )

        yield chunk

## 8. Historical grouping: from Geant4 steps to analysis-level charged-particle groups

This is the most important piece to understand if you want to modify the analysis.

The raw tree is step-level. The historical analysis combines related `e-`, `e+`, and `alpha` step rows using the pair:

```text
(Nucleus, ProcessType)
```

plus the historical ionization-continuation rule.

A group also keeps a dictionary keyed by `VolumeNumber`. For each gas volume it stores:

```text
[ summed_energy_in_that_volume,
  first_x,
  first_y,
  first_z ]
```

If further steps occur in the same volume, only the energy is accumulated; the first stored position is retained. The dictionary preserves insertion order, which matters because the fiducial cut later uses the **first volume / first position**.

The algorithm below is intentionally literal. Do not replace it with a track-ID based reconstruction if your goal is parity with the current background analysis.

In [ ]:
@dataclass
class Group:
    event: int
    particle: str
    nucleus: str
    process: str
    ionization_seen: bool = False
    volumes: dict = field(default_factory=dict)

    def add(self, volume, energy, x, y, z):
        if volume in self.volumes:
            # Same gas volume: add only the new deposited energy.
            self.volumes[volume][0] += energy
        else:
            # First time in this volume: retain this first position.
            self.volumes[volume] = [energy, x, y, z]


def groups(chunks):
    group = None
    fields = (
        "EventNumber", "ParticleName", "Nucleus", "ProcessType", "VolumeNumber",
        "EnergyDeposit", "x_hits", "y_hits", "z_hits",
    )

    for chunk in chunks:
        arrays = [chunk[name] for name in fields]

        for event, particle, nucleus, process, volume, energy, x, y, z in zip(*arrays):
            # A reconstructed group is never allowed to cross a Geant4 event boundary.
            if group is not None and event != group.event:
                yield group
                group = None

            # Historical processing only follows e-, e+ and alpha rows.
            if particle not in ("e-", "e+", "alpha"):
                continue

            ionization = process in ("ionIoni", "eIoni")

            if group is not None:
                if process == group.process and nucleus == group.nucleus and not group.ionization_seen:
                    # Preserve the inherited group's particle-label convention.
                    group.particle = str(particle)
                elif ionization:
                    # Once an ionization-created continuation appears, keep it inside this group.
                    group.ionization_seen = True
                else:
                    # Different historical group: emit the previous object first.
                    yield group
                    group = None

            if group is None:
                group = Group(
                    event=int(event),
                    particle=str(particle),
                    nucleus=str(nucleus),
                    process=str(process),
                )

            group.add(
                int(volume),
                float(energy),
                float(x),
                float(y),
                float(z),
            )

    # Critical EOF flush: do not lose the last non-empty group.
    if group is not None:
        yield group

## 9. Look at a few reconstructed groups

This cell intentionally stops after a handful of groups. It is useful for understanding the object that the spectrum code receives.

For each group, note that:

- `particle` is the historical final group label;
- `volumes` may contain one or several gas copy numbers;
- energy is still in **MeV** at this stage;
- each volume stores only its first x/y/z position.

In [ ]:
def first_n_groups(raw_path, job, row, n=5):
    output = []
    with uproot.open(raw_path, array_cache=None) as f:
        accounting, hits = root_header(f, expected_identity, job["requested_primaries"])
        iterator = groups(hit_chunks(hits, accounting["RequestedEvents"], config["layout"], STEP_SIZE))
        for g in iterator:
            output.append(g)
            if len(output) >= n:
                break
    return output


demo_groups = first_n_groups(example_raw, example_job, example_row, n=5)
for i, g in enumerate(demo_groups, 1):
    print(f"Group {i}")
    print("  event:   ", g.event)
    print("  particle:", g.particle)
    print("  nucleus: ", g.nucleus)
    print("  process: ", g.process)
    print("  ionization_seen:", g.ionization_seen)
    print("  volumes:")
    for volume, values in g.volumes.items():
        energy, x, y, z = values
        print(f"    volume {volume:3d}: E={energy:.6g} MeV, first position=({x:.3f}, {y:.3f}, {z:.3f}) mm")
    print()

## 10. Energy windows, fiducial cut and histogram binning

### Total deposited group energy

For an accepted electron or positron group:

\[
E_{\rm group}[\mathrm{keV}]
= \sum_{v} 1000\,E_v[\mathrm{MeV}].
\]

The multiplication by 1000 is applied to **each per-volume energy before summation**, matching the C++ historical implementation.

### Fiducial cut

Let `(x, y, z)` be the first stored position in the first gas volume of the group and `(x_c, y_c, z_c)` that volume's center. The group is accepted by the 20 mm cut when

\[
|x-x_c| \le L_x/2 - 20\,\mathrm{mm},\qquad
|y-y_c| \le L_y/2 - 20\,\mathrm{mm},\qquad
|z-z_c| \le L_z/2 - 20\,\mathrm{mm}.
\]

### Exact unbinned windows

The official analysis counts the following windows **before histogram binning**:

- `all`: every finite group energy, including underflow/overflow;
- `gt10`: `E > 10 keV`, no upper limit;
- `gt10_le400`: `10 < E <= 400 keV`;
- `underflow`: `E < 0 keV`;
- `in_range`: `0 <= E < 2000 keV`;
- `overflow`: `E >= 2000 keV`.

The plotted spectrum itself uses **900 bins from 0 to 2000 keV**.

In [ ]:
WINDOWS = ("all", "gt10", "gt10_le400", "underflow", "in_range", "overflow")

BOUNDARIES = {
    "all": "all finite energies, including underflow and overflow",
    "gt10": "E > 10 keV, no upper limit",
    "gt10_le400": "10 < E <= 400 keV",
    "underflow": "E < 0 keV",
    "in_range": "0 <= E < 2000 keV",
    "overflow": "E >= 2000 keV",
}

BINS = 900
MAX_ENERGY = 2000.0


def window_flags(energy):
    require(math.isfinite(energy), "Nonfinite group energy")
    return (
        True,
        energy > 10,
        10 < energy <= 400,
        energy < 0,
        0 <= energy < 2000,
        energy >= 2000,
    )


def fiducial(group, geometry):
    # IMPORTANT: use FIRST inserted volume and FIRST stored position.
    volume, (_, x, y, z) = next(iter(group.volumes.items()))
    center = geometry["gas_centers_mm"][str(volume)]

    return all(
        abs(pos - c) <= size / 2 - 20
        for pos, c, size in zip((x, y, z), center, geometry["gas_size_mm"])
    )


def histogram_bin(energy):
    """Match ROOT 6.40 TAxis::FindBin behavior used by the parity analysis."""
    if energy < 0:
        return 0
    if energy >= MAX_ENERGY:
        return BINS + 1

    width = MAX_ENERGY / BINS
    approximate = int(energy / width)
    return (
        1
        + approximate
        - int(energy < width * approximate)
        + int(width * (approximate + 1) <= energy)
    )

## 11. Accumulate one contribution

For each reconstructed group:

1. increment the number of processed historical groups;
2. keep only final `e-` / `e+` groups for the ER spectrum;
3. sum deposited energy across all gas volumes;
4. evaluate exact energy-window flags;
5. evaluate the 20 mm fiducial cut;
6. increment uncut counts/histogram;
7. if the group passes the cut, increment cut counts/histogram too.

The arrays have two rows:

```text
index 0 = no fiducial cut
index 1 = 20 mm fiducial cut
```

In [ ]:
def accumulate(group_iterator, geometry):
    counts = np.zeros((2, 6), dtype=np.int64)
    histograms = np.zeros((2, BINS + 2), dtype=np.int64)  # underflow + 900 bins + overflow
    processed = 0

    for group in group_iterator:
        processed += 1

        # Alphas participate in historical grouping but not the final ER spectrum.
        if group.particle not in ("e-", "e+"):
            continue

        # Preserve historical multiplication order: EACH volume is converted MeV -> keV first.
        energy_keV = 0.0
        for values in group.volumes.values():
            energy_keV += values[0] * 1000

        flags = window_flags(energy_keV)
        passes_cut = fiducial(group, geometry)
        bin_index = histogram_bin(energy_keV)

        # Always fill row 0; fill row 1 only when the fiducial cut passes.
        for cut_index in range(2 if passes_cut else 1):
            counts[cut_index] += flags
            histograms[cut_index, bin_index] += 1

    return counts, histograms, processed

## 12. Normalization: simulated groups → expected events/year

For a physical contribution `i`, the code uses

\[
w_i =
\frac{A_i\,Q_i\,(60\times60\times24\times365)}{N_{\rm generated,i}}.
\]

Where:

- `A_i` is the matrix activity in `Bq/kg` or `Bq/piece`;
- `Q_i` is the constructed component mass in kg or number of resistor pieces;
- `N_generated,i` comes from the ROOT run accounting, not from an assumed constant;
- one Bq is one decay per second;
- one analysis year is exactly 365 days.

A Monte Carlo group count `n_i` therefore becomes

\[
R_i = n_i\,w_i\quad [\mathrm{events/year}].
\]

**Crucially, this scaling is applied to every physical component/isotope contribution before detector categories are summed.**

In [ ]:
def scale_for(row, quantities, generated):
    require(type(generated) is int and generated > 0, "Invalid actual generated count")
    return (
        row["activity"]
        * quantity_for(row, quantities)
        * 60 * 60 * 24 * 365
        / generated
    )


# Show the scale for the example contribution before processing the complete campaign.
example_scale = scale_for(
    example_row,
    campaign["quantities"],
    example_job["accounting"]["GeneratedPrimaries"],
)

print("Example contribution:", example_row["id"])
print("Activity:", example_row["activity"], example_row["activity_unit"])
print("Normalization quantity:", quantity_for(example_row, campaign["quantities"]))
print("Generated primaries:", example_job["accounting"]["GeneratedPrimaries"])
print("Weight per accepted MC group [events/year]:", example_scale)

## 13. Process all 26 ROOT files

This is the long-running cell. It reproduces the numerical loop in `study/analyze.py`:

```text
for each matrix contribution:
    checksum raw ROOT
    open with uproot
    validate RunMetadata and RunAccounting
    stream Hits
    group steps
    accumulate ER counts and 900-bin spectra
    compute physical normalization scale
    verify raw ROOT did not change while being analyzed
```

The result for each source is stored as a tuple:

```text
(matrix row, normalization scale, exact-window counts, raw histogram counts)
```

No category summation happens yet.

In [ ]:
results = []
analysis_inputs = {}

for row in matrix["contributions"]:
    if row["id"] not in jobs:
        continue

    job, raw_path = jobs[row["id"]]
    checksum_before = sha256_file(raw_path)

    with uproot.open(raw_path, array_cache=None) as f:
        accounting, hits = root_header(
            f,
            expected_identity,
            job["requested_primaries"],
        )
        require(accounting == job["accounting"], "Raw/manifest accounting mismatch")

        group_iterator = groups(
            hit_chunks(
                hits,
                accounting["RequestedEvents"],
                config["layout"],
                STEP_SIZE,
            )
        )

        counts, histogram, processed_groups = accumulate(group_iterator, geometry)

        analysis_inputs[row["id"]] = {
            "path": str(raw_path.relative_to(CAMPAIGN_DIR)),
            "sha256": checksum_before,
            "accounting": accounting,
            "hit_rows": int(hits.num_entries),
            "processed_groups": processed_groups,
        }

    checksum_after = sha256_file(raw_path)
    require(checksum_after == checksum_before, "Raw file changed during analysis")

    scale = scale_for(
        row,
        campaign["quantities"],
        accounting["GeneratedPrimaries"],
    )

    results.append((row, scale, counts, histogram))
    print(f"Processed {row['id']:24s}  groups={processed_groups:10d}  scale={scale:.6g} events/year per MC group")

print("\nFinished contributions:", len(results))

## 14. Contribution-level diagnostic table

The official report ultimately aggregates detector categories, but for understanding and debugging it is very useful to first inspect every one of the 26 physical contributions.

The table below is derived from the **same counts and scales** used by the official analysis. It shows:

- assay activity;
- mass or piece count used in the normalization;
- actual generated-primary count;
- uncut and fiducial-cut Monte Carlo group counts;
- uncut and cut expected rates in events/year;
- cut survival fraction.

This table is diagnostic; it does not change the official category summation.

In [ ]:
contribution_table = []

for row, scale, counts, histogram in results:
    job = jobs[row["id"]][0]
    generated = job["accounting"]["GeneratedPrimaries"]
    quantity = quantity_for(row, campaign["quantities"])

    n_uncut = int(counts[0, WINDOWS.index("all")])
    n_cut = int(counts[1, WINDOWS.index("all")])
    rate_uncut = n_uncut * scale
    rate_cut = n_cut * scale

    contribution_table.append({
        "contribution": row["id"],
        "category": row["category"],
        "activity": row["activity"],
        "activity_unit": row["activity_unit"],
        "quantity": quantity,
        "generated": generated,
        "mc_uncut": n_uncut,
        "mc_cut": n_cut,
        "rate_uncut_y": rate_uncut,
        "rate_cut_y": rate_cut,
        "cut_survival": (rate_cut / rate_uncut) if rate_uncut > 0 else float("nan"),
    })


display(Markdown(markdown_table(
    contribution_table,
    [
        "contribution", "category", "activity", "activity_unit", "quantity",
        "generated", "mc_uncut", "mc_cut", "rate_uncut_y", "rate_cut_y", "cut_survival",
    ],
    formatters={
        "activity": lambda x: f"{x:.6g}",
        "quantity": lambda x: f"{x:.6g}",
        "rate_uncut_y": lambda x: f"{x:.6g}",
        "rate_cut_y": lambda x: f"{x:.6g}",
        "cut_survival": lambda x: f"{x:.3%}" if math.isfinite(x) else "—",
    },
)))

## 15. Sum normalized contributions into thesis detector categories

Only now do we combine components.

For example:

```text
GEMs = normalized GEMsCore contributions + normalized GEMsOuter contributions
```

and

```text
Field Cage = normalized RingSupports contributions + normalized RingStrips contributions
```

This order is essential because `GEMsCore` and `GEMsOuter`, or `RingSupports` and `RingStrips`, correspond to different assay materials and therefore different activities and normalization scales.

For statistical errors the current code treats each historical group count as Poisson and independent between jobs. For a contribution with count `n` and weight `w`, the stored variance is

\[
\sigma^2 = n w^2.
\]

These are Monte Carlo group-count errors only; they do **not** include assay or geometry uncertainties.

In [ ]:
def summarize(contributions):
    windows = []
    categories = {}
    totals = {}
    histograms = {}

    for row, scale, counts, histogram in contributions:
        for cut in (0, 1):
            for w, window in enumerate(WINDOWS):
                count = int(counts[cut, w])
                item = {
                    "contribution": row["id"],
                    "window": window,
                    "fiducial": cut,
                    "count": count,
                    "rate_per_year": count * scale,
                    "variance_per_year2": count * scale * scale,
                }
                windows.append(item)

                # Add this already-scaled contribution to its category and total.
                for target, label in ((categories, row["category"]), (totals, "total")):
                    aggregate = target.setdefault(
                        (label, window, cut),
                        {
                            "category": label,
                            "window": window,
                            "fiducial": cut,
                            "count": 0,
                            "rate_per_year": 0.0,
                            "variance_per_year2": 0.0,
                        },
                    )
                    for key in ("count", "rate_per_year", "variance_per_year2"):
                        aggregate[key] += item[key]

            # Category histograms are also summed only after each contribution is scaled.
            h, variance = histograms.setdefault(
                (row["category"], cut),
                (np.zeros(BINS + 2), np.zeros(BINS + 2)),
            )
            h += histogram[cut] * scale
            variance += histogram[cut] * scale * scale

    # Convert histogram counts/bin/year to density counts/keV/year for plotting.
    density = []
    width = MAX_ENERGY / BINS

    for (category, cut), (h, variance) in sorted(histograms.items()):
        density.append({
            "category": category,
            "fiducial": cut,
            "bins": [
                {
                    "low_keV": i * width,
                    "high_keV": (i + 1) * width,
                    "counts_per_keV_year": float(h[i + 1] / width),
                    "error_per_keV_year": float(math.sqrt(variance[i + 1]) / width),
                }
                for i in range(BINS)
            ],
            "underflow_per_year": float(h[0]),
            "overflow_per_year": float(h[-1]),
        })

    return {
        "windows": windows,
        "categories": list(categories.values()),
        "totals": list(totals.values()),
        "density": density,
        "units": "windows: counts/year; density: counts/keV/year",
        "boundaries": BOUNDARIES,
        "error_convention": "Poisson group counts; independent jobs; no assay/geometry uncertainties",
    }


spectra = summarize(results)
print("Category/window rows:", len(spectra["categories"]))
print("Total/window rows:", len(spectra["totals"]))
print("Category density spectra:", len(spectra["density"]))

## 16. Reproduce the background spectra — thesis Fig. 7.5 and Fig. 7.6 style

The current repository uses the following category order:

```text
Camera Lenses
Vessel
Camera Sensors
Cathodes
Resistors
GEMs
Field Cage
```

The stack is cumulative. `figure7.5` is the uncut spectrum and `figure7.6` is the same analysis after the 20 mm fiducial cut.

The y-axis is a density:

```text
counts / keV / year
```

and the x-axis covers 0–2000 keV. Underflow and overflow are included in exact full-range totals but are not visible in the plotted 0–2000 keV spectrum.

In [ ]:
CATEGORIES = (
    "Camera Lenses",
    "Vessel",
    "Camera Sensors",
    "Cathodes",
    "Resistors",
    "GEMs",
    "Field Cage",
)

# Same palette used by the repository reporting.py, retained for plot parity.
COLORS = (
    "#0072B2", "#999999", "#56B4E9", "#E69F00",
    "#CC79A7", "#D55E00", "#009E73",
)


def plot_background_stack(spectra, cut, title=None):
    items = {
        d["category"]: d
        for d in spectra["density"]
        if d["fiducial"] == cut
    }

    fig, ax = plt.subplots(figsize=(12, 7), layout="constrained")
    bottom = None
    positives = []

    for category, color in zip(CATEGORIES, COLORS):
        if category not in items:
            ax.plot([], [], color=color, label=category + " (missing)")
            continue

        bins = items[category]["bins"]
        edges = [b["low_keV"] for b in bins] + [bins[-1]["high_keV"]]
        values = np.array([b["counts_per_keV_year"] for b in bins])

        if bottom is None:
            bottom = np.zeros_like(values)

        ax.stairs(
            bottom + values,
            edges,
            baseline=bottom,
            fill=True,
            color=color,
            label=category,
        )
        bottom += values
        positives.extend(values[values > 0])

    if positives:
        ax.set_yscale("log")
        ax.set_ylim(min(positives) * 0.25, max(bottom) * 20)
    else:
        ax.set_ylim(0, 1)

    ax.set(
        xlim=(0, 2000),
        xlabel="Deposited energy [keV]",
        ylabel="counts / keV / year",
        title=title or (
            "Internal radioactivity: 20 mm fiducial cut"
            if cut else
            "Internal radioactivity: no fiducial cut"
        ),
    )
    ax.legend(fontsize=9)
    return fig, ax

In [ ]:
fig75, ax75 = plot_background_stack(
    spectra,
    cut=0,
    title="Internal radioactivity — no fiducial cut (Fig. 7.5 style)",
)
plt.show()

In [ ]:
fig76, ax76 = plot_background_stack(
    spectra,
    cut=1,
    title="Internal radioactivity — 20 mm fiducial cut (Fig. 7.6 style)",
)
plt.show()

## 17. Table 7.3-style detector-category contributions before and after the cut

The published comparison values embedded in the current repository are:

| Detector category | No cut [events/year] | 20 mm cut [events/year] |
|---|---:|---:|
| Camera Lenses | 35 | 16 |
| Vessel | 1187 | 465 |
| Camera Sensors | 4278 | 1687 |
| Cathodes | 9293 | 196 |
| Resistors | 12542 | 810 |
| GEMs | 30540 | 1769 |
| Field Cage | 313451 | 17738 |

The notebook now puts the newly calculated values and their Monte Carlo statistical errors next to those references. The reference values are **not** used anywhere in the normalization.

In [ ]:
REFERENCE_73 = dict(zip(
    CATEGORIES,
    (
        (35, 16),
        (1187, 465),
        (4278, 1687),
        (9293, 196),
        (12542, 810),
        (30540, 1769),
        (313451, 17738),
    ),
))

category_lookup = {
    (x["category"], x["window"], x["fiducial"]): x
    for x in spectra["categories"]
}

category_comparison = []
for category in CATEGORIES:
    uncut = category_lookup.get((category, "all", 0))
    cut = category_lookup.get((category, "all", 1))

    category_comparison.append({
        "category": category,
        "calc_uncut_y": uncut["rate_per_year"] if uncut else None,
        "mcerr_uncut": math.sqrt(uncut["variance_per_year2"]) if uncut else None,
        "ref_uncut_y": REFERENCE_73[category][0],
        "calc_cut_y": cut["rate_per_year"] if cut else None,
        "mcerr_cut": math.sqrt(cut["variance_per_year2"]) if cut else None,
        "ref_cut_y": REFERENCE_73[category][1],
        "survival": (
            cut["rate_per_year"] / uncut["rate_per_year"]
            if uncut and cut and uncut["rate_per_year"] > 0
            else float("nan")
        ),
    })


display(Markdown(markdown_table(
    category_comparison,
    [
        "category", "calc_uncut_y", "mcerr_uncut", "ref_uncut_y",
        "calc_cut_y", "mcerr_cut", "ref_cut_y", "survival",
    ],
    formatters={
        "calc_uncut_y": lambda x: f"{x:.6g}",
        "mcerr_uncut": lambda x: f"{x:.4g}",
        "calc_cut_y": lambda x: f"{x:.6g}",
        "mcerr_cut": lambda x: f"{x:.4g}",
        "survival": lambda x: f"{x:.3%}" if math.isfinite(x) else "—",
    },
)))

## 18. Table 7.2-style total rates in exact energy windows

The current repository reports three exact windows against the thesis references:

- `all` — all finite energies, including histogram underflow and overflow;
- `gt10` — `E > 10 keV`, with no upper limit;
- `gt10_le400` — `10 < E <= 400 keV`.

The reference values currently encoded in the repository are:

```text
all:          370000 no cut, 22600 with cut
gt10:         320000 no cut, 17700 with cut
gt10_le400:   330000 no cut, 17600 with cut
```

The fact that the published no-cut `(10,400]` value exceeds the published `>10` value is internally inconsistent, but the repository deliberately preserves the printed reference rather than silently correcting it.

In [ ]:
REFERENCE_72 = {
    "all": (370000, 22600),
    "gt10": (320000, 17700),
    "gt10_le400": (330000, 17600),
}

total_lookup = {
    (x["window"], x["fiducial"]): x
    for x in spectra["totals"]
}

table72 = []
for window, reference in REFERENCE_72.items():
    for cut in (0, 1):
        value = total_lookup.get((window, cut))
        table72.append({
            "window": window,
            "definition": spectra["boundaries"][window],
            "fiducial_mm": 20 * cut,
            "calculated_counts_per_year": value["rate_per_year"] if value else None,
            "mc_standard_error": math.sqrt(value["variance_per_year2"]) if value else None,
            "published_reference_counts_per_year": reference[cut],
        })


display(Markdown(markdown_table(
    table72,
    [
        "window", "definition", "fiducial_mm",
        "calculated_counts_per_year", "mc_standard_error",
        "published_reference_counts_per_year",
    ],
    formatters={
        "calculated_counts_per_year": lambda x: f"{x:.6g}",
        "mc_standard_error": lambda x: f"{x:.4g}",
    },
)))

## 19. Full exact-window table by detector category

The official report only formats the full-range category values as Table 7.3. For analysis work, it is useful to expose **all exact windows for every category**. This is still based on the same official `spectra['categories']` object and therefore introduces no new event-selection logic.

In [ ]:
full_category_windows = []
for category in CATEGORIES:
    for window in WINDOWS:
        uncut = category_lookup.get((category, window, 0))
        cut = category_lookup.get((category, window, 1))
        if uncut is None and cut is None:
            continue

        full_category_windows.append({
            "category": category,
            "window": window,
            "uncut_rate_y": uncut["rate_per_year"] if uncut else None,
            "cut_rate_y": cut["rate_per_year"] if cut else None,
            "uncut_mc_error": math.sqrt(uncut["variance_per_year2"]) if uncut else None,
            "cut_mc_error": math.sqrt(cut["variance_per_year2"]) if cut else None,
        })


display(Markdown(markdown_table(
    full_category_windows,
    ["category", "window", "uncut_rate_y", "cut_rate_y", "uncut_mc_error", "cut_mc_error"],
    formatters={
        "uncut_rate_y": lambda x: f"{x:.6g}",
        "cut_rate_y": lambda x: f"{x:.6g}",
        "uncut_mc_error": lambda x: f"{x:.4g}",
        "cut_mc_error": lambda x: f"{x:.4g}",
    },
)))

## 20. Sanity checks that are useful when modifying the notebook

A few identities should hold automatically:

1. For every category and cut, the sum of normalized contribution rates equals the category rate.
2. The sum over all categories equals the total rate.
3. The fiducial-cut rate should never exceed the corresponding uncut rate for the same exact selection.
4. For each contribution, `all = underflow + in_range + overflow` in raw group counts.

These checks are not replacements for physics validation, but they catch many accidental notebook edits immediately.

In [ ]:
# 1 & 2: total rate equals sum of category rates for every window/cut.
for window in WINDOWS:
    for cut in (0, 1):
        total = total_lookup[(window, cut)]["rate_per_year"]
        summed_categories = sum(
            category_lookup[(category, window, cut)]["rate_per_year"]
            for category in CATEGORIES
            if (category, window, cut) in category_lookup
        )
        require(
            math.isclose(total, summed_categories, rel_tol=1e-12, abs_tol=1e-12),
            f"Category sum mismatch for {window}, cut={cut}",
        )

# 3: cut cannot exceed uncut for the same category/window.
for category in CATEGORIES:
    for window in WINDOWS:
        if (category, window, 0) in category_lookup and (category, window, 1) in category_lookup:
            require(
                category_lookup[(category, window, 1)]["rate_per_year"]
                <= category_lookup[(category, window, 0)]["rate_per_year"] + 1e-12,
                f"Cut > uncut for {category}, {window}",
            )

# 4: exact all count partitions into underflow + in-range + overflow.
for row, scale, counts, histogram in results:
    for cut in (0, 1):
        all_count = counts[cut, WINDOWS.index("all")]
        partition = (
            counts[cut, WINDOWS.index("underflow")]
            + counts[cut, WINDOWS.index("in_range")]
            + counts[cut, WINDOWS.index("overflow")]
        )
        require(all_count == partition, f"Window partition mismatch: {row['id']}, cut={cut}")

print("All bookkeeping sanity checks passed.")

## 21. Optional: compare the total cut reduction numerically

This is a compact way to see what the 20 mm cut does to the complete internal-radioactivity background.

In [ ]:
uncut_total = total_lookup[("all", 0)]["rate_per_year"]
cut_total = total_lookup[("all", 1)]["rate_per_year"]

print(f"Full-range total before cut: {uncut_total:.6g} events/year")
print(f"Full-range total after  cut: {cut_total:.6g} events/year")
print(f"Survival fraction:           {cut_total / uncut_total:.3%}")
print(f"Rejection fraction:          {1 - cut_total / uncut_total:.3%}")

## 22. Export notebook results

The code below writes the main numerical products so that you can compare this notebook with the normal `study/analyze.py` output or use them in downstream studies.

The numerical `spectra.json` structure is the same conceptual object produced by the repository analysis. The CSV files additionally include the contribution-level diagnostic table created in this notebook.

In [ ]:
OUTPUT_DIR = Path("./cygno_notebook_analysis_output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# JSON needs normal Python types, which summarize() already provides for its public object.
(OUTPUT_DIR / "spectra.json").write_text(json.dumps(spectra, indent=2) + "\n")


def write_csv(path, rows):
    if not rows:
        return
    with Path(path).open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


write_csv(OUTPUT_DIR / "contributions_full.csv", contribution_table)
write_csv(OUTPUT_DIR / "table7.2.csv", table72)
write_csv(OUTPUT_DIR / "table7.3_comparison.csv", category_comparison)
write_csv(OUTPUT_DIR / "category_windows_full.csv", full_category_windows)

# Save the two figures using the same plotting function used above.
fig, _ = plot_background_stack(spectra, cut=0)
fig.savefig(OUTPUT_DIR / "figure7.5.png", dpi=160)
fig.savefig(OUTPUT_DIR / "figure7.5.pdf")
plt.close(fig)

fig, _ = plot_background_stack(spectra, cut=1)
fig.savefig(OUTPUT_DIR / "figure7.6.png", dpi=160)
fig.savefig(OUTPUT_DIR / "figure7.6.pdf")
plt.close(fig)

print("Wrote notebook outputs to:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" ", path.name)

## 23. Where to modify the analysis deliberately

The notebook is structured so that changes have a clear physical meaning:

### Change the grouping model
Edit **Section 8** (`groups`). This changes how raw Geant4 steps become analysis-level charged-particle objects. Such a change breaks strict parity with Samuele's historical grouping and should be treated as a new reconstruction version.

### Change the fiducial cut
Edit **Section 10** (`fiducial`). For example, change `20` mm to another margin or test all stored positions rather than the first one. Again, that is a new selection and will no longer reproduce the historical result exactly.

### Change energy windows
Edit **Section 10** (`WINDOWS`, `window_flags`). The exact windows are independent from the 900-bin plotting histogram.

### Change plotting binning
Edit `BINS` / `MAX_ENERGY`. This changes the displayed spectrum density but does not have to change exact-window counts.

### Change activity assumptions
Do **not** edit normalization numbers inside the spectrum loop. Modify the scientific source matrix instead, because the correct logic is always:

```text
activity for one physical contribution
× its own mass/piece count
× seconds/year
÷ its own generated-primary count
```

and only then sum categories.

### Change detector geometry / masses
Those values originate from the constructed simulation geometry and its exported `geometry.json` / `quantities.tsv`. Editing them only in this notebook would make analysis inconsistent with the Monte Carlo geometry.

## 24. Interpretation boundary

If this notebook reproduces the current repository results, that establishes **analysis-pipeline parity**. It does not by itself prove that the detector geometry, source-position model, radioassay inputs, or Geant4 radioactive-decay physics exactly reproduce every physical assumption in the thesis.

The output of this notebook is the internal-radioactivity background spectrum that forms the input to the later sensitivity study. The subsequent thesis steps — isotropic background angular template, pp solar-neutrino signal template, detector smearing, two-dimensional energy/angle likelihood, toy Monte Carlo, Bayesian evidence and discovery-probability curves — are a separate analysis stage and are not silently included here.